In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

class DataCleaner:
    def __init__(self, csv_filename):
        self.df = pd.read_csv(csv_filename)
        self.copy = self.df.copy()

    def clean_data(self):
        print("Cleaning data...")

        # --- 1. FILTERING NON-GAMES ---
        filter_keywords = ['Preseason', 'Hall Of Fame', 'BYE', 'CANCELLED', 'Pro Bowl']
        for keyword in filter_keywords:
            self.copy = self.copy[~self.copy['Week'].str.contains(keyword, na=False, case=False)]

        # Drop rows where Date is missing
        self.copy.dropna(subset=['Date'], inplace=True)
        self.copy.reset_index(drop=True, inplace=True)

        # --- 2. FIX DATES ---
        # We need 'Season' temporarily to figure out the correct year.
        date_strs = self.copy['Date'].astype(str) + '/' + self.copy['Season'].astype(str)

        # Convert to datetime objects
        self.copy['Date'] = pd.to_datetime(date_strs, format='%m/%d/%Y', errors='coerce')
        self.copy.dropna(subset=['Date'], inplace=True)

        # Fix Year for Playoff Games (Jan/Feb/Mar)
        # (e.g., A Playoff game in Season 2017 happening on Jan 5th is actually Jan 5th, 2018)
        mask = self.copy['Date'].dt.month <= 3
        self.copy.loc[mask, 'Date'] = self.copy.loc[mask, 'Date'] + pd.DateOffset(years=1)

        # --- 3. SPLIT RECORDS ---
        # Logic: Split "3-1" into separate integer columns

        # Away Team
        away_split = self.copy['AwayRecord'].astype(str).str.split('-', n=1, expand=True)
        if away_split.shape[1] == 2:
            self.copy['AwayWins'] = pd.to_numeric(away_split[0], errors='coerce').fillna(0).astype(int)
            self.copy['AwayLosses'] = pd.to_numeric(away_split[1], errors='coerce').fillna(0).astype(int)

        # Home Team
        home_split = self.copy['HomeRecord'].astype(str).str.split('-', n=1, expand=True)
        if home_split.shape[1] == 2:
            self.copy['HomeWins'] = pd.to_numeric(home_split[0], errors='coerce').fillna(0).astype(int)
            self.copy['HomeLosses'] = pd.to_numeric(home_split[1], errors='coerce').fillna(0).astype(int)

        # --- 4. LINK TEAM TO TOTAL RECORD ---
        # Calculate Win Percentages

        # Away Stats
        self.copy['AwayTotalGames'] = self.copy['AwayWins'] + self.copy['AwayLosses']
        self.copy['AwayWinPct'] = self.copy['AwayWins'] / self.copy['AwayTotalGames']
        self.copy['AwayWinPct'] = self.copy['AwayWinPct'].fillna(0.0).round(3)

        # Home Stats
        self.copy['HomeTotalGames'] = self.copy['HomeWins'] + self.copy['HomeLosses']
        self.copy['HomeWinPct'] = self.copy['HomeWins'] / self.copy['HomeTotalGames']
        self.copy['HomeWinPct'] = self.copy['HomeWinPct'].fillna(0.0).round(3)

        # --- 5. CLEANUP ---
        # Remove the Season column as requested
        self.copy.drop(columns=['Season'], inplace=True)

        print("Data cleaned successfully.")

    def save_and_download(self, output_filename="cleaned_nfl_data.csv"):
        self.copy.to_csv(output_filename, index=False)
        print(f"File saved as {output_filename}")
        files.download(output_filename)

# --- MAIN EXECUTION BLOCK ---
print("Please upload your CSV file (e.g., 2017-2025_scores.csv):")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    cleaner = DataCleaner(filename)
    cleaner.clean_data()
    cleaner.save_and_download()
else:
    print("No file uploaded.")

Please upload your CSV file (e.g., 2017-2025_scores.csv):


In [ ]:
from google.colab import drive
drive.mount('/content/drive')